# Notebook 01: Dataset Generation from CIC-IDS2017 PCAPs via Zeek

## Pipeline (per spec: PCAP → Zeek → Feature Extraction → ML Dataset)

1. **PCAP files** → Run through Zeek → generate conn.log, dns.log, http.log, ssl.log
2. **Zeek logs** → Parse with our log_parser → structured DataFrames
3. **Parsed logs** → feature_extractor → Zeek-based feature vectors
4. **CIC-IDS2017 CSVs** → Used ONLY for ground-truth labels (mapping by IP/timestamp)
5. **Zeek features + CSV labels** → Training/test/zero-day datasets

### Why PCAPs and not CSVs?
The CIC-IDS2017 CSVs contain features computed by CICFlowMeter (80+ columns). Our system uses **Zeek-extracted features** at inference time. Training on CICFlowMeter features would create a feature mismatch with deployment. We must train on the same features we extract at runtime.

### Dataset
- **PCAPs**: Download from https://www.unb.ca/cic/datasets/ids-2017.html → place in `data/pcaps/`
- **CSVs** (labels only): Download from same source → place in `data/csv/`

In [ ]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath('..'))
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

with open('../config/config.json', 'r') as f:
    config = json.load(f)

dataset_cfg = config['dataset']
PCAP_DIR = os.path.join('..', dataset_cfg['data_dir'], 'pcaps')
CSV_DIR = os.path.join('..', dataset_cfg['data_dir'], 'csv')
OUTPUT_DIR = os.path.join('..', dataset_cfg['data_dir'], 'processed')
os.makedirs(PCAP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

from src.zeek_runner import ZeekRunner
from src.log_parser import parse_all_logs
from src.feature_extractor import FeatureExtractor

zeek_runner = ZeekRunner(config)
feature_extractor = FeatureExtractor(config)

## Step 1: Locate PCAP and CSV files

- **data/pcaps/**: CIC-IDS2017 PCAP files (Monday through Friday)
- **data/csv/**: CIC-IDS2017 CSV label files (providing ground-truth labels per flow)

In [ ]:
pcap_files = sorted([f for f in os.listdir(PCAP_DIR) if f.endswith(('.pcap', '.pcapng'))])
csv_files = sorted([f for f in os.listdir(CSV_DIR) if f.endswith('.csv')])

print(f'PCAP files ({len(pcap_files)}):')
for f in pcap_files:
    size_mb = os.path.getsize(os.path.join(PCAP_DIR, f)) / (1024*1024)
    print(f'  {f} ({size_mb:.1f} MB)')

print(f'\nCSV label files ({len(csv_files)}):')
for f in csv_files:
    print(f'  {f}')

assert len(pcap_files) > 0, f'No PCAP files in {PCAP_DIR}. Download CIC-IDS2017 PCAPs.'
assert len(csv_files) > 0, f'No CSV files in {CSV_DIR}. Download CIC-IDS2017 CSVs for labels.'

## Step 2: Process each PCAP through Zeek → Extract Features

This is the core of the pipeline: each PCAP is fed to Zeek, logs are parsed, and features are extracted using our feature_extractor module - the **same** module used at inference time.

In [ ]:
all_features = []

for pcap_file in pcap_files:
    pcap_path = os.path.join(PCAP_DIR, pcap_file)
    pcap_name = os.path.splitext(pcap_file)[0]
    log_dir = os.path.join(OUTPUT_DIR, 'zeek_logs', pcap_name)
    os.makedirs(log_dir, exist_ok=True)

    print(f'\nProcessing {pcap_file}...')
    try:
        zeek_runner.run_offline(pcap_path, log_dir)
    except RuntimeError as e:
        print(f'  Zeek failed: {e}')
        continue

    log_paths = zeek_runner.get_log_paths(log_dir)
    print(f'  Zeek logs found: {list(log_paths.keys())}')

    if 'conn' not in log_paths:
        print(f'  No conn.log - skipping')
        continue

    logs = parse_all_logs(log_paths)
    features = feature_extractor.build_feature_vector(logs)

    if features.empty:
        print(f'  No features extracted - skipping')
        continue

    features['pcap_source'] = pcap_name
    all_features.append(features)
    print(f'  Extracted {len(features)} flows, {len(features.columns)} features')

combined = pd.concat(all_features, ignore_index=True)
print(f'\nTotal flows extracted: {len(combined)}')
print(f'Total features: {len([c for c in combined.columns if combined[c].dtype in (np.float64, np.int64, float, int)])}')

## Step 3: Load CIC-IDS2017 Labels

The CSV files provide **ground-truth labels only**. We do NOT use their feature columns. We match labels to Zeek flows by source/destination IP.

In [ ]:
dfs = []
for f in csv_files:
    path = os.path.join(CSV_DIR, f)
    df = pd.read_csv(path, low_memory=False, encoding_errors='replace')
    df.columns = df.columns.str.strip()
    dfs.append(df)
    print(f'Loaded {f}: {df.shape}')

cic_df = pd.concat(dfs, ignore_index=True)
print(f'\nCombined CSV shape: {cic_df.shape}')

# Find label column
label_col = None
for candidate in ['Label', 'label']:
    if candidate in cic_df.columns:
        label_col = candidate
        break
assert label_col, f'Label column not found. Columns: {list(cic_df.columns)}'

print(f'\nLabel distribution:')
print(cic_df[label_col].value_counts())

## Step 4: Map Labels & Assign to Zeek Flows

Map CIC-IDS2017 label strings to our 4 training + 3 excluded classes.
Match flows by (source_ip, destination_ip) pair.

In [ ]:
training_classes = dataset_cfg['training_classes']  # Benign, DoS, BruteForce, PortScan
excluded_classes = dataset_cfg['excluded_classes']  # DDoS, Botnet, WebAttack

def map_label(lbl):
    low = lbl.lower().strip()
    if low in ('benign', 'benvolent'):
        return 'Benign'
    elif 'ddos' in low:
        return 'DDoS'
    elif 'dos' in low:
        return 'DoS'
    elif 'patator' in low or 'brute' in low:
        return 'BruteForce'
    elif 'portscan' in low or 'port scan' in low:
        return 'PortScan'
    elif 'bot' in low:
        return 'Botnet'
    elif 'web' in low:
        return 'WebAttack'
    else:
        return 'Other'

cic_df['mapped_label'] = cic_df[label_col].apply(map_label)
print('Mapped label distribution:')
print(cic_df['mapped_label'].value_counts())

# Build lookup: (src_ip, dst_ip) -> mapped_label
src_ip_col = next((c for c in ['Source IP', ' Source IP', 'src_ip'] if c in cic_df.columns), None)
dst_ip_col = next((c for c in ['Destination IP', ' Destination IP', 'dst_ip'] if c in cic_df.columns), None)

assert src_ip_col, f'Source IP column not found. Available: {list(cic_df.columns)}'
assert dst_ip_col, f'Destination IP column not found. Available: {list(cic_df.columns)}'

label_lookup = {}
for _, row in cic_df[[src_ip_col, dst_ip_col, 'mapped_label']].dropna().iterrows():
    key = (str(row[src_ip_col]), str(row[dst_ip_col]))
    label_lookup[key] = row['mapped_label']

print(f'\nLabel lookup table entries: {len(label_lookup)}')

def assign_label(row):
    src = str(row.get('src_ip', row.get('id.orig_h', '')))
    dst = str(row.get('dst_ip', row.get('id.resp_h', '')))
    label = label_lookup.get((src, dst), 'Benign')
    return label

combined['mapped_label'] = combined.apply(assign_label, axis=1)
print('\nZeek flows with labels:')
print(combined['mapped_label'].value_counts())

# Binary label for Tier-1
combined['binary_label'] = combined['mapped_label'].apply(lambda x: 0 if x == 'Benign' else 1)
print('\nBinary distribution:')
print(combined['binary_label'].value_counts())

## Step 5: Select Zeek Features & Split Datasets

Only use features that come from our Zeek feature extractor (matching inference).

In [ ]:
non_feature = ['mapped_label', 'binary_label', 'pcap_source', 'uid', 'ts',
                'src_ip', 'dst_ip', 'id.orig_h', 'id.resp_h', 'id.orig_p', 'id.resp_p']
feature_cols = [c for c in combined.columns if c not in non_feature
                and combined[c].dtype in (np.float64, np.int64, float, int, np.int32, np.float32)]
print(f'Zeek features selected: {len(feature_cols)}')
print(f'Features: {feature_cols}')

# Handle categorical columns: one-hot encode service, conn_state, proto, method, tls_version, ja3_hash
categorical_cols = [c for c in ['service', 'conn_state', 'proto', 'method', 'tls_version', 'ja3_hash']
                    if c in combined.columns]
if categorical_cols:
    print(f'\nOne-hot encoding categorical columns: {categorical_cols}')
    combined = pd.get_dummies(combined, columns=categorical_cols, dummy_na=True)
    # Recompute feature columns after one-hot encoding
    feature_cols = [c for c in combined.columns if c not in non_feature
                    and combined[c].dtype in (np.float64, np.int64, float, int, np.int32, np.float32,
                                                 bool, np.bool_)]
    # Convert bool to int
    for c in feature_cols:
        if combined[c].dtype == bool:
            combined[c] = combined[c].astype(int)
    print(f'Features after encoding: {len(feature_cols)}')

# Clean
combined = combined.replace([np.inf, -np.inf], np.nan)
combined[feature_cols] = combined[feature_cols].fillna(0)

# Split
training_classes = dataset_cfg['training_classes']
excluded_classes = dataset_cfg['excluded_classes']

df_train = combined[combined['mapped_label'].isin(training_classes)].copy()
df_zero_day = combined[combined['mapped_label'].isin(excluded_classes)].copy()

print(f'\nTraining set: {len(df_train)} flows')
print(df_train['mapped_label'].value_counts())
print(f'\nZero-day set: {len(df_zero_day)} flows')
print(df_zero_day['mapped_label'].value_counts())

## Step 6: Train/Test Splits & Save

In [ ]:
X_all = df_train[feature_cols]
y_binary = df_train['binary_label']
y_multi = df_train['mapped_label']

# Tier-1: binary split
X_t1_train, X_t1_test, y_t1_train, y_t1_test = train_test_split(
    X_all, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
print(f'Tier-1 train: {X_t1_train.shape}, test: {X_t1_test.shape}')

# Tier-2: anomaly-only split
df_anomaly = df_train[df_train['binary_label'] == 1]
X_anomaly = df_anomaly[feature_cols]
y_anomaly = df_anomaly['mapped_label']

X_t2_train, X_t2_test, y_t2_train, y_t2_test = train_test_split(
    X_anomaly, y_anomaly, test_size=0.2, random_state=42, stratify=y_anomaly
)
print(f'Tier-2 train: {X_t2_train.shape}, test: {X_t2_test.shape}')

# Zero-day
X_zeroday = df_zero_day[feature_cols]
y_zeroday = df_zero_day['mapped_label']

# Save
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, 'feature_order.joblib'))

X_t1_train.to_csv(os.path.join(OUTPUT_DIR, 'tier1_X_train.csv'), index=False)
X_t1_test.to_csv(os.path.join(OUTPUT_DIR, 'tier1_X_test.csv'), index=False)
y_t1_train.to_csv(os.path.join(OUTPUT_DIR, 'tier1_y_train.csv'), index=False)
y_t1_test.to_csv(os.path.join(OUTPUT_DIR, 'tier1_y_test.csv'), index=False)

X_t2_train.to_csv(os.path.join(OUTPUT_DIR, 'tier2_X_train.csv'), index=False)
X_t2_test.to_csv(os.path.join(OUTPUT_DIR, 'tier2_X_test.csv'), index=False)
y_t2_train.to_csv(os.path.join(OUTPUT_DIR, 'tier2_y_train.csv'), index=False)
y_t2_test.to_csv(os.path.join(OUTPUT_DIR, 'tier2_y_test.csv'), index=False)

X_zeroday.to_csv(os.path.join(OUTPUT_DIR, 'zeroday_X.csv'), index=False)
y_zeroday.to_csv(os.path.join(OUTPUT_DIR, 'zeroday_y.csv'), index=False)

combined.to_csv(os.path.join(OUTPUT_DIR, 'all_features_labeled.csv'), index=False)

print(f'\nDataset saved to {OUTPUT_DIR}/')
print(f'Feature count: {len(feature_cols)}')
print(f'Tier-1 train: {X_t1_train.shape}')
print(f'Tier-2 train: {X_t2_train.shape}')
print(f'Zero-day eval: {X_zeroday.shape}')
print(f'\nFeatures: {feature_cols}')